# 🎧 Extrair Texto de Áudio ou Vídeo

Transcreve qualquer arquivo de áudio ou vídeo para texto usando **faster-whisper** (Whisper da OpenAI otimizado).  
100% gratuito, roda localmente, sem API key.

**Formatos suportados:** MP3, MP4, WAV, M4A, OGG, WEBM, MKV, AVI, MOV, FLAC

**Gera:**
- `.txt` — texto puro da narração
- `.srt` — legenda com timecodes (importar no Camtasia)

## 1. Instalar dependências

In [5]:
# !pip install faster-whisper

In [6]:
import torch

# Verificar se a GPU está disponível
print("PyTorch versão:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nome da GPU:", torch.cuda.get_device_name(0))
    print("Memória da GPU:", torch.cuda.memory_allocated(0), "bytes")

PyTorch versão: 2.7.1+cu118
CUDA disponível: True
Nome da GPU: NVIDIA RTX A2000 12GB
Memória da GPU: 0 bytes


## 2. Configuração — informe o arquivo e escolha o modelo

In [7]:
from pathlib import Path

# ============================================================
# CONFIGURAÇÕES — edite aqui
# ============================================================

# Caminho do arquivo de entrada (áudio ou vídeo)
# Exemplos:
#   '../outputs/audio/narracao_imovel_feminina.mp3'
#   'C:/Videos/meu_video.mp4'
#   '../outputs/audio/gravacao_propria.wav'
ARQUIVO_ENTRADA = r'D:\Projetos\prompt2promolab\outputs\vido1.mp4'

# Modelo Whisper:
#   'tiny'     → mais rápido, qualidade menor   (bom para testes)
#   'base'     → rápido, qualidade boa
#   'small'    → equilibrado ← recomendado para PT-BR
#   'medium'   → lento, qualidade alta
#   'large-v3' → máxima qualidade, bem lento
MODELO = 'small'

# Idioma: 'pt' para Português, 'en' para Inglês
# None → detecta automaticamente
IDIOMA = 'pt'

# Pasta de saída (None = mesma pasta do arquivo de entrada)
PASTA_SAIDA = Path('../outputs/legendas')

# ============================================================

arquivo = Path(ARQUIVO_ENTRADA)
pasta_saida = PASTA_SAIDA if PASTA_SAIDA else arquivo.parent
pasta_saida.mkdir(parents=True, exist_ok=True)

print(f'Arquivo : {arquivo}')
print(f'Modelo  : {MODELO}')
print(f'Idioma  : {IDIOMA or "auto-detectar"}')
print(f'Saída   : {pasta_saida}')

if not arquivo.exists():
    print('\n⚠️  Arquivo não encontrado! Verifique o caminho em ARQUIVO_ENTRADA.')
else:
    import os
    kb = os.path.getsize(arquivo) / 1024
    print(f'\n✅ Arquivo encontrado ({kb:.1f} KB)')

Arquivo : D:\Projetos\prompt2promolab\outputs\vido1.mp4
Modelo  : small
Idioma  : pt
Saída   : ..\outputs\legendas

✅ Arquivo encontrado (10309.1 KB)


## 3. Transcrever

In [ ]:
import torch
from faster_whisper import WhisperModel

# Detecta GPU automaticamente — usa CUDA se disponível, senão CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'

if device == 'cuda':
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}')
else:
    print('CUDA não disponível — usando CPU')

print(f'Device       : {device}')
print(f'Compute type : {compute_type}')
print('Carregando modelo Whisper (primeira vez faz download automático)...')

model = WhisperModel(MODELO, device=device, compute_type=compute_type)

print('Transcrevendo... aguarde.\n')
segments_iter, info = model.transcribe(
    str(arquivo),
    language=IDIOMA,
    beam_size=5,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=400),
    word_timestamps=False,
)

segmentos = list(segments_iter)

print(f'Idioma detectado : {info.language} (confiança: {info.language_probability:.0%})')
print(f'Duração          : {info.duration:.1f}s')
print(f'Segmentos        : {len(segmentos)}')

GPU detectada: NVIDIA RTX A2000 12GB
Device       : cuda
Compute type : float16
Carregando modelo Whisper (primeira vez faz download automático)...
Transcrevendo... aguarde.



## 4. Salvar texto puro (.txt)

In [ ]:
texto_puro = '\n'.join(seg.text.strip() for seg in segmentos)

arquivo_txt = pasta_saida / f'{arquivo.stem}_transcricao.txt'
arquivo_txt.write_text(texto_puro, encoding='utf-8')

print(f'✅ Texto salvo: {arquivo_txt}')
print('\n' + '='*60)
print('TRANSCRIÇÃO COMPLETA')
print('='*60)
print(texto_puro)
print('='*60)

## 5. Salvar legenda SRT (.srt)

In [ ]:
def formatar_tempo_srt(segundos: float) -> str:
    h  = int(segundos // 3600)
    m  = int((segundos % 3600) // 60)
    s  = int(segundos % 60)
    ms = int((segundos % 1) * 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

blocos = []
for i, seg in enumerate(segmentos, start=1):
    inicio = formatar_tempo_srt(seg.start)
    fim    = formatar_tempo_srt(seg.end)
    blocos.append(f'{i}\n{inicio} --> {fim}\n{seg.text.strip()}')

conteudo_srt = '\n\n'.join(blocos) + '\n'

arquivo_srt = pasta_saida / f'{arquivo.stem}.srt'
arquivo_srt.write_text(conteudo_srt, encoding='utf-8')

print(f'✅ Legenda SRT salva: {arquivo_srt}')
print('\n--- Prévia do SRT ---')
print(conteudo_srt[:600])

## 6. Resumo dos arquivos gerados

In [ ]:
import os

print('Arquivos gerados:')
for f in [arquivo_txt, arquivo_srt]:
    kb = os.path.getsize(f) / 1024
    print(f'  {f}  ({kb:.1f} KB)')

print()
print('Próximos passos no Camtasia Studio 8:')
print('  1. Edit > Captions > Import Captions → selecione o .srt')
print('  2. As legendas entram sincronizadas automaticamente na timeline')